# 2 - Process and Merge Datasets
Next, we want to process the ADCP and Science Datasets so that we can merge them into a single unified dataset, derive the individual profile dives, and run the vendor-suggested Quality Control algorithms.

In [ ]:
import os

import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
# For this example we will be using CP05MOAS-GL379 D00008, which was deployed at the Pioneer NES array from 2019-09-27 to 2019-11-19.
refdes = 'CP05MOAS-GL379-01-ADCPAM000'
eng_refdes = 'CP05MOAS-GL379-00-ENG000000'
deployment = 8

In [ ]:
dep_str = f"{deployment:04d}"  # always 4 digits, e.g., '0008'
glider_name = '-'.join(refdes.split('-')[:2]) # shorten ref des to get glider name

In [ ]:
# Load the ADCP Data
adcp = xr.open_dataset(f"../data/raw/{refdes}.deployment{dep_str}.adcp_data.nc").load()

# Load the Science Data
gdac = xr.open_dataset(f"../data/raw/{glider_name}.deployment{dep_str}.science.nc").load()

#### Merge the Datasets
Merging the glider ADCP and Science datasets is a multistep process. The Science dataset contains all of the different science sensors, along with the glider's built-in sensors and navigation, on a common timestamp. However, the different science sensors all sample at different rates. Thus, the science parameters in the dataset are sparse and filled with NaNs. In order to merge the Science and ADCP data streams, we need to do the following:
1. Split the separate science and glider parameters into different datasets based on their source sensor
2. Drop the NaNs from each science parameter, so each one is interpolated from only its own measurements
3. Linearly interpolate each science parameter to the ADCP time. ADCP times inside a gap in a parameter's measurements longer than `max_gap` (60 seconds by default), e.g. while the sensor is off, are left as NaN rather than interpolated across
4. Merge the interpolated, grouped parameter sets to the ADCP dataset

The interpolated CTD measurements (`ctd_temperature`, `ctd_salinity`, `ctd_pressure`, `ctd_density`, ...) are the ones to use with the ADCP data. The ADCP's own `temperature` is from the temperature probe on its transducer, which is less accurate and precise than the glider CTD, its `salinity` is a constant (35) set in its configuration, and its `pressure` is empty: the depth of the ADCP measurements (`depth_from_pressure`) is already calculated by OOINet from the glider CTD pressure.

Two corrections are first made to the OOINet ADCP data:
- The ADCP heading, pitch, and roll are reported in hundredths of a degree, but OOINet labels them as degrees, so they are converted to degrees with `adcp_attitude_to_degrees`.
- OOINet rotates the ADCP water velocities by the magnetic declination (about 14 degrees), assuming they are relative to magnetic north. The glider ADCP has no compass and uses the glider's heading, which is already relative to true north, so this is a double correction. `true_north_velocities` builds `eastward/northward_seawater_velocity` from the raw velocities (`water_velocity_east/north`) instead, and keeps OOINet's rotated velocities as `ooi_eastward/northward_seawater_velocity` for reference (see notebook 4 for the check against the GPS).

In [ ]:
from ooicgsn_glider_dvl.merge import (
    adcp_attitude_to_degrees,
    add_waypoints,
    downcast,
    merge_datasets,
    netcdf_encoding,
    true_north_velocities,
)

In [ ]:
# Convert the ADCP heading, pitch, and roll to degrees, and use the raw (true north) velocities
adcp = adcp_attitude_to_degrees(adcp)
adcp = true_north_velocities(adcp)

merged = merge_datasets(adcp, gdac)

# Store the (time, bin) ADCP variables as float32 to halve the memory use
merged = downcast(merged)
merged

In [ ]:
# Compare the ADCP's own temperature and salinity with the interpolated glider CTD
adcp_temperature = merged['temperature'] / 100  # hundredths of a degree C
difference = (adcp_temperature - merged['ctd_temperature']).to_series().dropna()
print(f"ADCP salinity values: {np.unique(merged['salinity'])}")
print(f"CTD salinity range: {float(merged['ctd_salinity'].min()):.2f} to {float(merged['ctd_salinity'].max()):.2f}")
print(f"ADCP - CTD temperature (degrees C): median {difference.median():.2f}, "
      f"5th to 95th percentile {difference.quantile(0.05):.2f} to {difference.quantile(0.95):.2f}")
print(f"ADCP times without CTD data: {int(merged['ctd_temperature'].isnull().sum())} "
      f"of {merged.sizes['time']}")

#### Identify Individual Profiles/Dives
The ADCP on OOI gliders is only active during dives. The algorithm to separate the profiles looks for where the absolute difference between two depths exceeds 2 meters and the sampling interval is greater than 4 seconds (the ADCP samples at 2 seconds). Both thresholds can be changed with the `min_depth_change` and `min_time_gap` arguments of `add_profiles`.

In [ ]:
# Next, identify the individual profiles
from ooicgsn_glider_dvl.profiles import add_profiles

In [ ]:
merged = add_profiles(merged)
merged

### Vendor QA/QC 

First, utilize the TRDI Data QA-QC Model rev12-1 thresholds to derive a first-pass QC flag. We assume that the Explorer DVL 600kHz is comparable to the WH600kHz (Wide-Band) for setting the thresholds.
I think the config settings are: 2.0 meter depth cell size, 0.1 ping interval time, 10 pings per ensemble

| Parameter | Good | Suspect | Bad |
| --------- | ---- | ------- | --- |
| Error Velocity Threshold | <=6.3 cm/s | >6.3 cm/s, <=12.6 cm/s | >=12.6 cm/s |
| Correlation Magnitude | >=115 | <115, >=63 | <63 |
| Percent Good | >=63% | <63%, >=50% | <50% |
| BIT Errors | 0 | >=1 | N/A |

In [ ]:
from ooicgsn_glider_dvl import qc

In [ ]:
# Run the individual tests (make sure inputs and units match)
percent_good = qc.percent_good_qc(merged, 63, 50)
corr_mag = qc.correlation_magnitude_qc(merged, 115, 63)
error_vel = qc.error_velocity_qc(merged, 6.3/100, 12.6/100)

# Merge the test results
qc_flag = qc.merge_qc([percent_good, corr_mag, error_vel])

In [ ]:
qc_flag

In [ ]:
# Now add the qc_flags to the dataset
merged['vendor_qc_flag'] = (['time','bin'], qc_flag)
merged['vendor_qc_flag'].attrs = {
            'long_name': 'TRDI QC Summary Flag',
            'standard_name': 'aggregate_quality_flag',
            'comment': ('Summary of the TRDI QC tests as a QARTOD style summary flag, where '
                        'the values are 1 == pass, 2 == not evaluated, 3 == suspect or of high interest, '
                        '4 == fail, and 9 == missing.'),
            'flag_values': np.array([1, 2, 3, 4, 9], dtype=np.int8),
            'flag_meanings': 'pass not_evaluated suspect_or_of_high_interest fail missing'
        }

#### Add Waypoints
The next step is to add the waypoint data into the merged dataset. The waypoint data is included in the glider engineering data and is not in either the science or adcp datasets. The waypoint data is necessary to calculate the net water-column velocity, which is achieved by comparing the expected waypoint with the actual surfacing location. Each waypoint applies from the time it is set until the next waypoint is set. A waypoint of (0, 0) is the glider's placeholder before a waypoint is set (e.g. after a reboot), so ADCP times while it applies have no waypoint (NaN).

In [ ]:
# Use the recovered engineering data, or the telemetered data if the recovered data wasn't available
eng_file = f"../data/raw/deployment{dep_str}_{eng_refdes}-recovered_host-glider_eng_recovered.csv"
if not os.path.exists(eng_file):
    eng_file = f"../data/raw/deployment{dep_str}_{eng_refdes}-telemetered-glider_eng_telemetered.csv"
    print(f"No recovered engineering data, using the telemetered data: {eng_file}")

# Only the waypoint, dead-reckoning, GPS, flight (pitch, heading, buoyancy, speed), and depth
# columns are needed from the engineering data
from ooicgsn_glider_dvl.currents import CURRENT_VARIABLES
from ooicgsn_glider_dvl.flight import FLIGHT_VARIABLES

columns = set(['c_wpt_lat', 'c_wpt_lon', 'm_depth'] + CURRENT_VARIABLES + FLIGHT_VARIABLES)
eng = pd.read_csv(eng_file, usecols=lambda c: c in columns, parse_dates=['time'])
eng.head()

In [ ]:
merged = add_waypoints(merged, eng)

#### Add the Depth-Averaged Current
Between surfacings, the glider makes many dives and dead reckons its position from its heading, its speed through the water, and the current it estimated at the previous surfacing. When it surfaces, the displacement between its first GPS fix and its dead-reckoned position is the drift that the dead reckoning did not account for. Dividing the displacement by the time spent underwater gives the correction to the current estimate, and adding it to the current used in the dead reckoning gives the depth-averaged current over all of the dives since the last surfacing.

The positions are the glider's own dead-reckoned surfacing position (`m_dr_surf_x/y_lmc`) and first GPS fix (`m_gps_fix_x/y_lmc`), in meters east and north. The glider's `m_lat` and `m_lon` can't be used for this, because after surfacing they ramp from the dead-reckoned position to the GPS fix.

Each ADCP time between the start of the dives and the surfacing is given the depth-averaged current calculated at that surfacing.

In [ ]:
from ooicgsn_glider_dvl.currents import add_depth_averaged_current, surfacing_currents

# Calculate the depth-averaged current at each surfacing
surfacings = surfacing_currents(eng)
surfacings.head()

In [ ]:
merged = add_depth_averaged_current(merged, surfacings)

#### Build the Glider Dataset
The ADCP is only on during the dives, so the merged ADCP dataset doesn't cover the climbs or the surfacings. The glider flight model, though, is best calibrated with both the dives and the climbs, since the buoyancy changes sign between them, and the depth-averaged current from the GPS is an average over all of the dives and climbs since the last surfacing. So we also build a glider dataset on the time base of the glider engineering data, which the glider records throughout (about every 4 s):
1. The glider depth (`m_depth`, from the glider's own pressure sensor) is kept as it is.
2. The science variables are interpolated to the engineering times in the same way as for the ADCP (`glider_dataset`), e.g. `ctd_pressure`, `ctd_temperature`, `ctd_salinity`, and `ctd_density`. The science data only covers the dives and climbs, so they are NaN at the surface.
3. The glider pitch, heading, buoyancy change, and its own estimate of its speed through the water are added from the engineering data (`add_flight_variables`), as `vehicle_pitch`, `vehicle_heading`, `buoyancy_change`, and `dead_reckoning_speed`.
4. The waypoints and depth-averaged currents are added in the same way as for the ADCP.

In [ ]:
from ooicgsn_glider_dvl.flight import add_flight_variables
from ooicgsn_glider_dvl.merge import glider_dataset

glider = glider_dataset(eng[['time', 'm_depth']], gdac)
glider['m_depth'].attrs.update({'long_name': 'Glider Depth', 'units': 'm'})
glider = add_flight_variables(glider, eng)
glider = add_waypoints(glider, eng)
glider = add_depth_averaged_current(glider, surfacings)
glider

#### Label the Dives, Climbs, and Surfacings
The science (IOOS GDAC) dataset is split into profiles, alternating between dives and climbs, which we list with `science_profiles`. A profile is a dive if the CTD pressure at its end is greater than at its start. `add_phases` then labels each time in the glider dataset with its science profile (`science_profile_id`) and phase:
- **dive** or **climb**: from the start to the end of a science profile
- **surface**: in a gap between two profiles that lasts at least `min_surface_time` (60 s by default). The climbs end a few meters below the surface, and the dives that follow start at the surface.
- **unlabeled**: the shorter gaps (about 1 s) at the inflections between a dive and a climb, or between a climb and a dive that don't surface, and the times before the first or after the last profile

As a check, each surfacing should match a surfacing in the depth-averaged currents, which are calculated from the engineering data independently of the science profiles.

In [ ]:
from ooicgsn_glider_dvl.profiles import PHASES, add_phases, science_profiles

profiles = science_profiles(gdac)
glider = add_phases(glider, profiles)
print(profiles['phase'].value_counts().to_string(), '\n')

# The number of glider measurements in each phase and their depths
phase_names = {value: key for key, value in PHASES.items()}
glider_phases = pd.DataFrame({
    'phase': pd.Series(glider['phase'].values).map(phase_names),
    'depth': glider['m_depth'].values,
})
print(glider_phases.groupby('phase')['depth'].describe(percentiles=[0.05, 0.5, 0.95]).round(1))

In [ ]:
# Each surfacing in the glider dataset should match a surfacing in the depth-averaged
# currents: the glider reports its dead-reckoned surfacing position shortly after the
# climb ends
at_surface = (glider['phase'] == PHASES['surface']).values.astype(int)
surface_start = glider['time'].values[np.flatnonzero(np.diff(at_surface, prepend=0) == 1)]
nearest = np.abs(surfacings['surface_time'].values[:, None] - surface_start[None, :]).min(axis=1)
nearest = pd.Series(nearest / np.timedelta64(1, 's'))
print(f'{surface_start.size} surfacings in the glider dataset, '
      f'{len(surfacings)} depth-averaged currents')
print(f'Time from the start of the nearest surfacing to the dead-reckoned surfacing (s): '
      f'median {nearest.median():.0f}, maximum {nearest.max():.0f}')
surfacings[nearest > 120]

#### Link the ADCP Dives to the Science Dives
Each ADCP profile is linked to the science profile it overlaps the most in time (`link_profiles`). The ADCP turns on a few seconds before a dive that starts at a shallow inflection, and a couple of minutes after a dive that starts at the surface, so the ADCP and science profiles don't start and end at exactly the same times. Then `add_profile_links` adds the linked `science_profile_id` and `phase` to each ADCP measurement, and the linked ADCP `adcp_profile_id` to each glider measurement in a science profile, including the parts of the dive before the ADCP turned on.

In [ ]:
from ooicgsn_glider_dvl.profiles import add_profile_links, link_profiles

links = link_profiles(merged, profiles)
dives = profiles.loc[profiles['phase'] == 'dive', 'science_profile_id']
print(f"{len(links)} ADCP profiles linked to {links['science_profile_id'].nunique()} "
      f"science profiles: {(links['phase'] == 'dive').sum()} dives and "
      f"{(links['phase'] == 'climb').sum()} climbs")
print(f"Science dives without an ADCP profile: {(~dives.isin(links['science_profile_id'])).sum()}")
print(f"Science profiles with more than one ADCP profile: "
      f"{links['science_profile_id'].duplicated().sum()}")
print(f"Fraction of each ADCP profile overlapping its science profile: "
      f"median {links['overlap_fraction'].median():.3f}, "
      f"minimum {links['overlap_fraction'].min():.3f}")
links.describe()

In [ ]:
# The ADCP profiles linked to a climb: a few measurements near the end of the climb
links[links['phase'] == 'climb']

In [ ]:
merged, glider = add_profile_links(merged, glider, links)

In [ ]:
# Save the results, compressing each variable
os.makedirs("../data/processed", exist_ok=True)
merged.to_netcdf(
    f"../data/processed/{glider_name}.deployment{dep_str}.merged.nc",
    format='netcdf4',
    engine='h5netcdf',
    encoding=netcdf_encoding(merged),
)

# Save the glider dataset through the dives, climbs, and surfacings
glider.to_netcdf(
    f"../data/processed/{glider_name}.deployment{dep_str}.glider.nc",
    format='netcdf4',
    engine='h5netcdf',
    encoding=netcdf_encoding(glider),
)

# Save the depth-averaged current at each surfacing, the science profiles, and the links
# between the ADCP and science profiles
surfacings.to_csv(f"../data/processed/{glider_name}.deployment{dep_str}.depth_averaged_current.csv", index=False)
profiles.to_csv(f"../data/processed/{glider_name}.deployment{dep_str}.science_profiles.csv", index=False)
links.to_csv(f"../data/processed/{glider_name}.deployment{dep_str}.profile_links.csv")